# 2. Príznaky snímok pomocou ResNet18

Na 272 klipoch nie je možné natrénovať konvolučnú sieť od nuly, preto použijeme
**transfer learning**: sieť **ResNet18** predtrénovanú na ImageNet (1,2 mil. obrázkov, 1000 tried).

- Poslednú vrstvu `fc` (klasifikácia do 1000 tried ImageNet) nahradíme `nn.Identity()`.
  Výstupom je potom vektor **512 príznakov** na snímku (výstup global average pooling).
- Sieť je **zmrazená** – netrénujeme ju, len ňou prevedieme snímky na príznaky.
- Snímky normalizujeme rovnako, ako pri trénovaní ResNet18 na ImageNet (`to_imagenet_tensor`).

Výsledok: `features_resnet.npy` s tvarom `(počet klipov, 16, 512)`.
Tieto príznaky používajú modely **CNN-SVM** (notebook 4) a **CNN-LSTM** (notebook 5).

In [ ]:
import numpy as np
import pandas as pd
import torch
from torch import nn
from torchvision.models import resnet18, ResNet18_Weights

from har_utils import read_frames, to_imagenet_tensor

df = pd.read_csv("index.csv")
device = "cuda" if torch.cuda.is_available() else "cpu"
print(len(df), "klipov,", device)

In [ ]:
model = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)
model.fc = nn.Identity()       # namiesto 1000 tried ImageNet -> 512 príznakov
model.eval().to(device)        # eval: BatchNorm a Dropout v režime inferencie

In [ ]:
feats = []
with torch.no_grad():                                      # bez gradientov: rýchlejšie, menej pamäte
    for path in df.path:
        x = to_imagenet_tensor(read_frames(path)).to(device)   # (16, 3, 224, 224)
        feats.append(model(x).cpu().numpy())                   # (16, 512)

feats = np.stack(feats)
np.save("features_resnet.npy", feats)
print(feats.shape)